#### Tools

Models ca nrequest to call tools that perfrom tasks such as ferching data from a databasse searching the web, or running code. tools are pairinngs of:

    1. A schema, including the name of the tool, , a description, and/or argument defination (aften a json schema)
    2. A function or coroutine to execute

#### what is halucination ?

when model has not knowledge of any question or answer how ever it try to give answer and it produce unaccurate answer.
e.g., model trained on data of may 2025 but you will ask coding  question such as is this correct syntax as per latest documentation it will produce answer but not accurate 

#### How to reduce halucination?

==> we can use tools so llm call this tools and take contact from tool and then generate output with prompt.

#### what is reAct agent?
tool calling like on question it call tools line by line such as like when you ask any latest question it will search n google and if you ask like what is 5 +5 it will call calculator tool so called as reAct agent.






In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:llama-3.1-8b-instant",max_tokens=100)

In [2]:
response = model.invoke("write me an essay on AI")
response

AIMessage(content="**The Rise of Artificial Intelligence: Opportunities and Challenges**\n\nArtificial Intelligence (AI) has become a ubiquitous term in today's technology-driven world. From virtual assistants like Siri and Alexa to self-driving cars and personalized recommendations on social media, AI has permeated every aspect of our lives. The rapid advancement of AI has led to widespread excitement and concern about its potential impact on society. In this essay, we will explore the opportunities and challenges presented by AI, and examine the potential future of this technology.\n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 41, 'total_tokens': 141, 'completion_time': 0.148031059, 'completion_tokens_details': None, 'prompt_time': 0.005534898, 'prompt_tokens_details': None, 'queue_time': 0.051848938, 'total_time': 0.153565957}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_

In [3]:
stream = model.stream("Write me an essay on AI")

for chunk in stream:
    print(chunk.content, end="", flush=True)

**The Rise of Artificial Intelligence: A New Era of Human-Machine Interaction**

Artificial intelligence (AI) has become an integral part of our daily lives, transforming the way we live, work, and interact with one another. From virtual assistants like Siri and Alexa to self-driving cars and medical diagnosis systems, AI has made significant strides in recent years, revolutionizing various industries and domains. In this essay, we will explore the concept of AI, its applications, benefits, and challenges, as well

In [4]:
inputs = [
    "Write a short essay on AI",
    "Explain machine learning in simple terms",
    "What are the risks of AI?"
]

responses = model.batch(inputs)

for i, res in enumerate(responses):
    print(f"\nResponse {i+1}:\n{res.content}")


Response 1:
**The Evolution and Impact of Artificial Intelligence**

Artificial Intelligence (AI) has revolutionized the world, transforming the way we live, work, and interact with one another. From its early beginnings in the 1950s to the present day, AI has undergone significant advancements, driven by the convergence of technological innovations, computational power, and data availability.

The concept of AI was first proposed by Alan Turing in his 1950 paper "Computing Machinery and Intelligence," where he introduced the idea of

Response 2:
**What is Machine Learning?**

Machine learning is a way for computers to learn and improve their performance on a task without being explicitly programmed. It's like teaching a child to recognize pictures of animals. You show the child many pictures of different animals and say which one is a cat or dog. Over time, the child learns to recognize the animals on their own without you telling them.

**How Does Machine Learning Work?**

Here are 

In [5]:
## Tools 
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """GEt the weather at a location"""
    return f"it's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])


In [6]:
response=model_with_tools.invoke("what's the weather in Bangalore")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': '34gt45zag', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.02304372, 'completion_tokens_details': None, 'prompt_time': 0.012902724, 'prompt_tokens_details': None, 'queue_time': 0.050704942, 'total_time': 0.035946444}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019df135-d8d0-75d2-aa6a-6020ebb052a1-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': '34gt45zag', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}


In [7]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Bangalore'},
  'id': '34gt45zag',
  'type': 'tool_call'}]

#### Tools Execution Loop

In [8]:
## Step 1: Model generate tools calls
messages = [{"role": "user","content":"what's the weather in Banglore"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2 : Execute tools and collect result
for tool_call in ai_msg.tool_calls:
    # print(tool_call)
    tool_result = get_weather.invoke(tool_call)
    # print(tool_result)
    messages.append(tool_result)

## Step 3: Pass result back to model for final response
final_resposne = model_with_tools.invoke(messages)
print(final_resposne.text)

Note: The final response is not a function call and is just an example response.


In [9]:
messages

[{'role': 'user', 'content': "what's the weather in Banglore"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'f89d6j19f', 'function': {'arguments': '{"location":"Banglore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 221, 'total_tokens': 236, 'completion_time': 0.026765037, 'completion_tokens_details': None, 'prompt_time': 0.013506271, 'prompt_tokens_details': None, 'queue_time': 0.051627799, 'total_time': 0.040271308}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019df135-d974-7540-87ce-40d5ef1d1a5b-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Banglore'}, 'id': 'f89d6j19f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 221, 'output_tokens': 15, 'total_tokens': 236}),
 ToolMessage(content="it

In [10]:
ai_msg.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Banglore'},
  'id': 'f89d6j19f',
  'type': 'tool_call'}]